# Notebook 01 — Phase 2 agents

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI (Financial Services Industry) Semantic Layer Workshop on AWS — Workshop 2

---

Why does Phase 2 need different agents than Phase 1?

This notebook answers that question with working code. By the end, you will have
seen the structural differences between Phase 1 and Phase 2 agents — and you will
understand why behavioral signals require a fundamentally different agent posture
than deterministic query routing.

## Key terms for this notebook

| Term | What it is |
|------|------------|
| **Behavioral signal** | A pattern detected over time — such as declining login frequency or reduced portfolio review — that indicates a change in client engagement. Unlike a wealth signal (a single event), a behavioral signal requires session-level data from the LGD. |
| **EngagementDecay** | A behavioral signal type that fires when a client's interaction frequency drops below a threshold over a rolling window. Requires LGD session data, not just SLGD entity relationships. |
| **NetworkInfluence** | A behavioral signal type that fires when a client's network connections (household members, business partners) show correlated wealth events. Requires graph traversal across the LGD. |
| **LGD (Lexical Graph Database)** | The second tier of the two-tier Neptune architecture. Contains session-level, temporal, and probabilistic data that the SLGD does not carry. Phase 2 agents query the LGD for behavioral patterns. |
| **SLGD (Semantic Layer Graph Database)** | The first tier — deterministic, SHACL-validated entity data. Phase 1 agents query only the SLGD. |
| **Agent posture** | A declared property of an agent descriptor that states whether the agent is deterministic-audited, deterministic-unaudited, or probabilistic-guarded. Phase 2 introduces agents with postures that Phase 1 did not need. |
| **Conversational-context-manager** | A Phase 2 agent that maintains session state using AgentCore Memory, enabling multi-turn conversations where follow-up questions reference prior context. |

## Behavioral signals need a different kind of agent

Phase 1 agents are deterministic query routers. They receive a question, select a
SPARQL template from a validated library, execute it against the SLGD, and return
the result. The SLGD contains entity-level data: customers, households, advisory
relationships, wealth signals. Every query against the SLGD is a point-in-time
lookup — it asks what exists right now, not what changed over time. That is
sufficient for Phase 1's referral workflow, where the question is always "which
customers match these criteria today?"

Phase 2 introduces behavioral signals — EngagementDecay and NetworkInfluence —
that cannot be answered by a point-in-time lookup. EngagementDecay requires
comparing a client's session frequency over a rolling 90-day window against their
historical baseline. NetworkInfluence requires traversing the client's relationship
graph in the LGD to find correlated wealth events among connected entities. Both
patterns require data that lives in the LGD, not the SLGD, because the LGD carries
session-level temporal data that the SLGD's SHACL shapes do not admit. A Phase 1
agent that queries only the SLGD cannot detect these signals — it lacks access to
the data tier where the evidence lives.

The second structural difference is conversational memory. Phase 1 agents are
stateless: each invocation is independent, with no reference to prior interactions.
A Wealth Advisor using the Phase 2 UI needs to ask follow-up questions — "Of those,
which have the highest AUM?" — where "those" refers to the result of the previous
question. This requires an agent that maintains session-scoped context across
invocations: the conversational-context-manager. This agent uses AgentCore Memory
to store and retrieve conversation state within a session, enabling multi-turn
interactions that Phase 1's stateless agents cannot support.

These are not incremental additions to Phase 1 agents. They are structurally
different: they query a different data tier (LGD vs SLGD), they carry different
postures (some are probabilistic-guarded because behavioral signal detection
involves statistical thresholds), and one of them maintains state across calls.
Phase 2 therefore registers three new agents — behavioral-signal-agent,
conversational-context-manager, and theme-summarizer — each with capabilities
that no Phase 1 agent possesses.

In [ ]:
import sys
import os
import json

# Workshop 1's shared helpers.
sys.path.insert(0, "../../../agentic-semantic-layer/notebooks/shared")

from pathlib import Path

SPEC_DIR = "../../spec/04-aws-agent-registry"

def load_descriptors(subdir):
    path = os.path.join(SPEC_DIR, subdir)
    descriptors = []
    if not os.path.isdir(path):
        print(f"WARNING: {path} not found.")
        return descriptors
    for f in sorted(os.listdir(path)):
        if f.endswith(".json"):
            with open(os.path.join(path, f)) as fh:
                descriptors.append(json.load(fh))
    return descriptors

agent_descriptors = load_descriptors("agents")
phase_1_agents = [d for d in agent_descriptors if d.get("phase") == 1]
phase_2_agents = [d for d in agent_descriptors if d.get("phase") == 2]

print(f"Total agents loaded:  {len(agent_descriptors)}")
print(f"Phase 1 agents:       {len(phase_1_agents)}")
print(f"Phase 2 agents:       {len(phase_2_agents)}")
print()
print("Phase 2 agent names:")
for d in phase_2_agents:
    print(f"  - {d['agent_name']} (posture: {d.get('posture', 'unknown')})")
print("\nSetup complete.")

In [ ]:
# Build cell 1 — Compare postures between Phase 1 and Phase 2 agents.
#
# Phase 1 agents are all deterministic-audited or deterministic-unaudited.
# Phase 2 introduces probabilistic-guarded postures for behavioral signals.

print("Phase 1 agent postures:")
print("-" * 50)
p1_postures = set()
for d in phase_1_agents:
    posture = d.get("posture", "unknown")
    p1_postures.add(posture)
    print(f"  {d['agent_name']:<35} {posture}")

print()
print("Phase 2 agent postures:")
print("-" * 50)
p2_postures = set()
for d in phase_2_agents:
    posture = d.get("posture", "unknown")
    p2_postures.add(posture)
    print(f"  {d['agent_name']:<35} {posture}")

print()
new_postures = p2_postures - p1_postures
print(f"Postures unique to Phase 1: {p1_postures - p2_postures}")
print(f"Postures unique to Phase 2: {new_postures}")
print(f"Shared postures:            {p1_postures & p2_postures}")

In [ ]:
# Build cell 2 — Demonstrate behavioral-signal-agent detecting engagement decay.
#
# The behavioral-signal-agent queries the LGD for session-level data.
# Here we simulate the detection logic: compare recent session count
# against historical baseline to identify engagement decay.

# Simulated LGD session data for a client
client_sessions = {
    "client_uri": "atlas:client-rachel-kim",
    "baseline_sessions_per_month": 12,
    "recent_90d_sessions": 4,
    "rolling_window_days": 90,
}

def detect_engagement_decay(session_data, threshold=0.5):
    """Detect EngagementDecay from LGD session data.
    
    Fires when recent session rate drops below threshold
    of the historical baseline.
    """
    baseline_per_90d = session_data["baseline_sessions_per_month"] * 3
    recent = session_data["recent_90d_sessions"]
    decay_ratio = recent / baseline_per_90d if baseline_per_90d > 0 else 1.0
    
    return {
        "signal_type": "EngagementDecay",
        "client_uri": session_data["client_uri"],
        "decay_ratio": round(decay_ratio, 3),
        "threshold": threshold,
        "fired": decay_ratio < threshold,
        "data_source": "LGD",
    }

result = detect_engagement_decay(client_sessions)
print("Engagement decay detection result:")
print(json.dumps(result, indent=2))
print()
print(f"Signal fired: {result['fired']}")
print(f"Data source:  {result['data_source']} (not SLGD)")

In [ ]:
# Build cell 3 — Show that behavioral-signal-agent depends on LGD access.
#
# Phase 1 agents depend on atlas-sparql-mcp with graph_tier=slgd.
# The behavioral-signal-agent must query graph_tier=lgd.

print("MCP server dependencies by phase:")
print("=" * 55)

print("\nPhase 1 agents:")
for d in phase_1_agents:
    deps = d.get("dependencies", {}).get("mcp_servers", [])
    print(f"  {d['agent_name']:<35} → {deps}")

print("\nPhase 2 agents:")
for d in phase_2_agents:
    deps = d.get("dependencies", {}).get("mcp_servers", [])
    graph_tiers = d.get("dependencies", {}).get("graph_tiers", [])
    print(f"  {d['agent_name']:<35} → {deps}")
    if graph_tiers:
        print(f"  {'':35}   graph_tiers: {graph_tiers}")

## Verification

Two properties must hold: Phase 2 agents exist in the registry with the correct
count, and the behavioral-signal-agent is structurally configured to query the LGD
(not just the SLGD). If either fails, Phase 2 cannot detect the behavioral signals
that differentiate it from Phase 1.

In [ ]:
# Verification cell 1 — Phase 2 agents exist in the registry.

print("Verifying Phase 2 agent registration...")
print()

expected_p2_names = {
    "behavioral-signal-agent",
    "conversational-context-manager",
    "theme-summarizer",
}

actual_p2_names = {d["agent_name"] for d in phase_2_agents}

print(f"Expected Phase 2 agents: {sorted(expected_p2_names)}")
print(f"Actual Phase 2 agents:   {sorted(actual_p2_names)}")
print()

missing = expected_p2_names - actual_p2_names
if missing:
    print(f"VERIFICATION FAILED: Missing Phase 2 agents: {missing}")
    print("Check that agent descriptors in spec/04-aws-agent-registry/agents/")
    print("include phase: 2 for the behavioral-signal-agent,")
    print("conversational-context-manager, and theme-summarizer.")

assert len(phase_2_agents) >= 3, (
    f"Expected at least 3 Phase 2 agents, found {len(phase_2_agents)}. "
    "Ensure agent descriptors have phase: 2."
)
assert expected_p2_names.issubset(actual_p2_names), (
    f"Missing Phase 2 agents: {missing}. "
    "Check spec/04-aws-agent-registry/agents/ for phase: 2 descriptors."
)

print(f"[PASS] All {len(expected_p2_names)} expected Phase 2 agents are registered.")

In [ ]:
# Verification cell 2 — behavioral-signal-agent queries LGD, not just SLGD.

print("Verifying behavioral-signal-agent data tier access...")
print()

bsa = next(
    (d for d in phase_2_agents if d["agent_name"] == "behavioral-signal-agent"),
    None
)

if bsa is None:
    print("VERIFICATION FAILED: behavioral-signal-agent not found.")
    # Remediation: ensure the agent descriptor exists with phase: 2
    assert False, "behavioral-signal-agent descriptor missing from registry."

graph_tiers = bsa.get("dependencies", {}).get("graph_tiers", [])
mcp_deps = bsa.get("dependencies", {}).get("mcp_servers", [])

print(f"Agent:        {bsa['agent_name']}")
print(f"Graph tiers:  {graph_tiers}")
print(f"MCP servers:  {mcp_deps}")
print()

queries_lgd = "lgd" in graph_tiers or "LGD" in str(graph_tiers)

if not queries_lgd:
    print("VERIFICATION FAILED: behavioral-signal-agent does not declare LGD access.")
    print("The agent must include 'lgd' in dependencies.graph_tiers to access")
    print("session-level data required for EngagementDecay and NetworkInfluence.")
    print("Update the agent descriptor at:")
    print("  spec/04-aws-agent-registry/agents/behavioral-signal-agent.json")

assert queries_lgd, (
    "behavioral-signal-agent must declare LGD access in dependencies.graph_tiers. "
    "Behavioral signals require session-level data from the LGD tier."
)

print("[PASS] behavioral-signal-agent declares LGD access.")
print("Behavioral signals (EngagementDecay, NetworkInfluence) can be detected.")

## What just changed

You have seen why Phase 2 requires structurally different agents. Phase 1 agents
are stateless, deterministic query routers against the SLGD. Phase 2 adds agents
that query the LGD for temporal behavioral data, maintain conversational memory
across invocations, and carry probabilistic-guarded postures for statistical
threshold detection.

The next notebook explores the most novel of these capabilities: AgentCore Memory,
which gives the conversational-context-manager the ability to remember what was
said earlier in a session — enabling the multi-turn interactions that a Wealth
Advisor's workflow demands.